# Codex connection test successful

CHB-MIT project starter notebook. This checks the GitHub-to-Colab code workflow; extraction and entropy production rate analysis are not implemented yet.

In [ ]:
print("CHB-MIT project notebook is working!")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/CHB_MIT_Project")

for folder in [
    "raw_segments",
    "metadata",
    "preprocessed",
    "results",
    "logs",
]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

print("Project folder ready:", PROJECT)

# Extract raw EEG segments
Run the cells above first to mount Drive and set PROJECT. This section downloads only EDF headers and selected data records using HTTP Range; it stops if range requests are not supported. Tiny surrounding record-boundary samples may be transferred and then trimmed.

Start with chb01. After reviewing its report, change CASES to all 24 cases. No filtering, padding, added noise, or cross-file pre/post extension is performed. Every annotated seizure is retained in full. Pre/post windows are up to 30 seconds, clipped to file edges and neighbouring seizures. Overlapping post/pre windows split the intervening gap at its midpoint, so samples are not duplicated between labels.

Comparison policy: candidate interictal duration equals preictal + ictal + postictal duration, matched per compatible channel/calibration layout. Candidates exclude all annotated seizures and BASELINE_GUARD seconds on either side within their source file. The default guard is 30 seconds: an operational exclusion rule, NOT proof of a clinical interictal period. Other recordings from the same case may supply comparison data; pre/post never borrow from another file. The script does not check time distance to seizures in adjacent recordings. Review the protocol with your supervisor before entropy analysis.

Candidates are NOT certified normal, awake, active, or artifact-free. No sleep/wake or artifact classification is implemented. Selection is deterministic from the first eligible intervals, not random or matched on sleep state.

NPZ files preserve int16 digital samples separately for each original channel, including repeated labels/dummy channels. The metadata carries units, calibration, channel labels, source interval and sampling rates. Convert with physical = (digital-digital_min)*(physical_max-physical_min)/(digital_max-digital_min)+physical_min before physical-unit analysis. Do not analyse dummy channels.

Combined outputs are separated by label and channel/calibration layout. Their metadata preserves every join. Do not filter or calculate time-dependent features across artificial joins. A baseline shortage is reported, never filled with repeated/synthetic data.

One writer per case. On interruption, rerun: completed segment files with valid sample counts are reused. Combined outputs are regenerated. This is extraction only; EPR requires an agreed estimator.

Sources: https://physionet.org/content/chbmit/1.0.0/ and https://www.edfplus.info/specs/edf.html


In [ ]:
"""CHB-MIT partial EDF extraction. Raw digital samples; no filtering or padding."""
import json, re, hashlib, math, io
from pathlib import Path
import urllib.request
import numpy as np

BASE = 'https://physionet.org/files/chbmit/1.0.0'
VERSION = '1'

def byte_range(url, start, end):
    req = urllib.request.Request(url, headers={'Range': f'bytes={start}-{end}', 'Accept-Encoding':'identity'})
    with urllib.request.urlopen(req, timeout=90) as r:
        cr = r.headers.get('Content-Range', '')
        if r.status != 206 or not cr.startswith(f'bytes {start}-{end}/'):
            raise RuntimeError(f'Partial download unavailable: HTTP {r.status}, {cr}. No full-file fallback.')
        data = r.read(end-start+2)
    if len(data) != end-start+1:
        raise ValueError('Incomplete byte range; rerun to retry.')
    return data

def header(url):
    first = byte_range(url, 0, 255)
    size = int(first[184:192]); n = int(first[252:256])
    if size != 256*(n+1): raise ValueError('Unexpected EDF header size')
    h = first + byte_range(url, 256, size-1)
    if h[192:236].strip().startswith(b'EDF+D'): raise ValueError('Discontinuous EDF unsupported')
    pos = 256
    out = {}
    for name, width, cast in [('labels',16,str),('transducers',80,str),('units',8,str),('physical_min',8,float),('physical_max',8,float),('digital_min',8,int),('digital_max',8,int),('prefilters',80,str),('samples_per_record',8,int),('reserved',32,str)]:
        vals = [h[pos+i*width:pos+(i+1)*width].decode('ascii').strip() for i in range(n)]
        out[name] = [cast(v) for v in vals]; pos += n*width
    out.update(header_bytes=size, records=int(h[236:244]), record_seconds=float(h[244:252]), url=url)
    if out['records'] < 0 or out['record_seconds'] <= 0: raise ValueError('Invalid EDF length')
    out['duration'] = out['records']*out['record_seconds']
    out['record_bytes'] = 2*sum(out['samples_per_record'])
    return out

def read_interval(h, start, end):
    """Fetch enclosing EDF records, then trim to [start,end) on each channel's sample grid."""
    d = h['record_seconds']; a = math.floor(start/d); b = math.ceil(end/d)
    begin = h['header_bytes'] + a*h['record_bytes']
    stop = h['header_bytes'] + b*h['record_bytes']
    parts = [byte_range(h['url'], p, min(p+4*1024*1024,stop)-1) for p in range(begin,stop,4*1024*1024)]
    records = np.frombuffer(b''.join(parts),dtype='<i2').reshape(b-a,-1)
    signals = {}; offset = 0
    for i,n in enumerate(h['samples_per_record']):
        fs = n/d
        lo,hi = round(start*fs), round(end*fs)
        if abs(lo-start*fs)>1e-6 or abs(hi-end*fs)>1e-6: raise ValueError('Interval not aligned to sample grid')
        signals[f'ch_{i:03d}'] = records[:,offset:offset+n].reshape(-1)[lo-a*n:hi-a*n].copy()
        assert len(signals[f'ch_{i:03d}']) == hi-lo
        offset += n
    return signals

def parse_summary(text):
    files=[]
    for block in re.split(r'File Name:\s*',text)[1:]:
        filename=block.splitlines()[0].strip()
        count=re.search(r'Number of Seizures in File:\s*(\d+)',block)
        if not count: raise ValueError(f'Missing seizure count: {filename}')
        starts=[float(v) for v in re.findall(r'Seizure(?:\s+\d+)? Start Time:\s*(\d+(?:\.\d+)?)',block)]
        ends=[float(v) for v in re.findall(r'Seizure(?:\s+\d+)? End Time:\s*(\d+(?:\.\d+)?)',block)]
        if len(starts)!=int(count[1]) or len(ends)!=len(starts): raise ValueError(f'Annotation parse mismatch: {filename}')
        seizures=list(zip(starts,ends))
        if any(a>=b for a,b in seizures) or any(seizures[i][1]>seizures[i+1][0] for i in range(len(seizures)-1)): raise ValueError('Invalid seizure ordering')
        files.append((filename,seizures))
    if not files: raise ValueError('No EDF entries parsed')
    return files

def merge(intervals):
    out=[]
    for a,b in sorted(intervals):
        if b<=a: continue
        if out and a<=out[-1][1]: out[-1][1]=max(b,out[-1][1])
        else: out.append([a,b])
    return out

def complement(intervals,duration):
    out=[]; cursor=0
    for a,b in merge(intervals):
        a=max(0,a); b=min(duration,b)
        if cursor<a: out.append((cursor,a))
        cursor=max(cursor,b)
    if cursor<duration: out.append((cursor,duration))
    return out

def plan_events(seizures,duration,window):
    rows=[]
    for i,(a,b) in enumerate(seizures):
        if not 0<=a<b<=duration: raise ValueError('Seizure outside EDF')
        # If surrounding windows overlap between seizures, split the gap at its midpoint.
        left=0 if i==0 else (seizures[i-1][1]+a)/2
        right=duration if i==len(seizures)-1 else (b+seizures[i+1][0])/2
        for label,s,e in [('preictal',max(0,a-window,left),a),('ictal',a,b),('postictal',b,min(duration,b+window,right))]:
            rows.append(dict(label=label,seizure=i+1,start=s,end=e,duration=e-s))
    return rows

def signature(h):
    # Include calibration: concatenated raw integer values must keep the same meaning.
    keys=['labels','units','samples_per_record','record_seconds','physical_min','physical_max','digital_min','digital_max']
    return hashlib.sha256(json.dumps({k:h[k] for k in keys},sort_keys=True).encode()).hexdigest()[:12]

def save_json(path,obj):
    tmp=path.with_suffix(path.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2)); tmp.replace(path)

def extract_case(project,case,window=30,baseline_guard=30):
    """Baseline equals total event-segment duration, matched within channel/calibration group.
    baseline_guard is seconds on each side of every annotated seizure IN THE SOURCE FILE.
    It does not establish a clinical interictal interval across file boundaries.
    """
    if not re.fullmatch(r'chb(?:0[1-9]|1[0-9]|2[0-4])',case): raise ValueError('Invalid case')
    if window<0 or baseline_guard<window: raise ValueError('Guard must be at least pre/post window')
    root=Path(project)/'raw_segments'/f'extraction_v{VERSION}_w{window}_g{baseline_guard}'/case
    root.mkdir(parents=True,exist_ok=True)
    summary_url=f'{BASE}/{case}/{case}-summary.txt'
    with urllib.request.urlopen(summary_url,timeout=90) as r: summary=r.read().decode('utf-8-sig')
    (root/'source_summary.txt').write_text(summary)
    files=parse_summary(summary)
    headers={}; rows=[]; targets={}; remaining={}
    def get_header(filename):
        if filename not in headers: headers[filename]=header(f'{BASE}/{case}/{filename}')
        return headers[filename]
    def store_segment(filename,h,row):
        row=dict(row,source_file=filename,group=signature(h))
        key=f"{Path(filename).stem}_{row['label']}_{row['start']:012.4f}_{row['end']:012.4f}"
        row['path']=key+'.npz' if row['duration']>0 else None
        if row['duration']>0:
            dest=root/row['path']
            valid=False
            if dest.exists():
                try:
                    with np.load(dest,allow_pickle=False) as z:
                        valid=all(len(z[f'ch_{i:03d}'])==round(row['duration']*n/h['record_seconds']) for i,n in enumerate(h['samples_per_record']))
                except Exception: valid=False
            if not valid:
                data=read_interval(h,row['start'],row['end'])
                meta=dict(row,edf=h,case=case,version=VERSION)
                tmp=dest.with_suffix('.tmp')
                with tmp.open('wb') as f: np.savez_compressed(f,metadata=np.array(json.dumps(meta)),**data)
                tmp.replace(dest)
        rows.append(row)
        save_json(root/'progress.json',{'case':case,'saved_segments':len(rows),'last':row})
    for filename,seizures in files:
        if not seizures: continue
        print(case,filename,'seizures:',len(seizures),flush=True)
        h=get_header(filename); group=signature(h)
        for row in plan_events(seizures,h['duration'],window):
            store_segment(filename,h,row)
            targets[group]=targets.get(group,0)+row['duration']
    remaining=dict(targets)
    # Same seizure-containing files first; then other files in this case if necessary.
    for filename,seizures in sorted(files,key=lambda x:not bool(x[1])):
        if not any(v>1e-8 for v in remaining.values()): break
        h=get_header(filename); group=signature(h)
        if remaining.get(group,0)<=1e-8: continue
        banned=[(max(0,a-baseline_guard),min(h['duration'],b+baseline_guard)) for a,b in seizures]
        for a,b in complement(banned,h['duration']):
            length=min(b-a,remaining[group])
            if length<=1e-8: continue
            store_segment(filename,h,dict(label='interictal_candidate',seizure=None,start=a,end=a+length,duration=length))
            remaining[group]-=length
    save_json(root/'manifest.json',rows)
    # Physical concatenation per class and compatible channel/calibration layout.
    # Boundaries retained: NEVER filter/analyse across these joins.
    for group in targets:
        for label in ['preictal','ictal','postictal','interictal_candidate']:
            selected=[r for r in rows if r['group']==group and r['label']==label and r['path']]
            if not selected: continue
            chunks={}; offsets={}; boundaries=[]
            for row in selected:
                with np.load(root/row['path'],allow_pickle=False) as z:
                    bounds={}
                    for k in z.files:
                        if k=='metadata': continue
                        data=z[k]; start=offsets.get(k,0)
                        chunks.setdefault(k,[]).append(data.copy()); offsets[k]=start+len(data)
                        bounds[k]=[start,offsets[k]]
                    edf=json.loads(str(z['metadata']))['edf']
                boundaries.append(dict(source=row,sample_bounds=bounds))
            dest=root/f'combined_{label}_{group}.npz'; tmp=dest.with_suffix('.tmp')
            with tmp.open('wb') as f:
                np.savez_compressed(f,metadata=np.array(json.dumps(dict(case=case,label=label,edf=edf,boundaries=boundaries))),**{k:np.concatenate(v) for k,v in chunks.items()})
            tmp.replace(dest)
    report={'case':case,'window_seconds':window,'baseline_guard_seconds':baseline_guard,'baseline_policy':'same-case candidate; excludes annotated seizures and guard within each source file; not certified normal/awake/artifact-free','total_seconds':{label:sum(r['duration'] for r in rows if r['label']==label) for label in ['preictal','ictal','postictal','interictal_candidate']},'baseline_shortfall_seconds_by_group':remaining,'channel_groups':list(targets),'overlap_policy':'split inter-seizure gap at midpoint when 30-second windows overlap','folder':str(root)}
    save_json(root/'report.json',report)
    print(json.dumps(report,indent=2),flush=True)
    return report


In [ ]:
CASES = ["chb01"]  # Validate this case before changing to all 24.
# CASES = [f"chb{i:02d}" for i in range(1, 25)]
PRE_POST_SECONDS = 30
BASELINE_GUARD = 30  # Operational minimum; not a clinical definition of normal EEG.
print("Cases:", CASES)
print("Output:", PROJECT / "raw_segments")


In [ ]:
reports = []
failures = {}
for case in CASES:
    try:
        reports.append(extract_case(PROJECT, case, window=PRE_POST_SECONDS, baseline_guard=BASELINE_GUARD))
    except Exception as exc:
        failures[case] = str(exc)
        print(f"FAILED {case}: {exc}. Completed segments remain saved; rerun after resolving the error.")
print("Successful cases:", [r["case"] for r in reports])
print("Failed cases:", failures)


In [ ]:
import pandas as pd
if reports:
    display(pd.DataFrame([dict(case=r["case"], **r["total_seconds"], baseline_shortfall=sum(r["baseline_shortfall_seconds_by_group"].values())) for r in reports]))
    print("Review report.json and manifest.json in each reported output folder.")
    print("For chb01 with 30-second windows, expect 7 seizures: 442 seconds ictal, 210 preictal, 210 postictal; verify against the saved source summary.")
else:
    print("No case completed. Inspect the failure messages above.")
